In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
RANDOM_SEED = 42

def set_seeds(seed: int=RANDOM_SEED) -> None:
    """Sets random sets for torch operations.

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed(RANDOM_SEED)

# 1. Get Data

In [6]:
import os
import zipfile

from pathlib import Path

import requests

def download_data(source: str, 
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                      destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

[INFO] Did not find data\pizza_steak_sushi directory, creating one...
[INFO] Downloading pizza_steak_sushi.zip from https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip...


BadZipFile: File is not a zip file

# 2. Get dataloaders

In [7]:
from going_modular import engine, data_setup
from pathlib import Path
import torchvision.transforms.v2 as T
import torchvision

DATA_DIR = Path("data/")
IMAGE_DIR = DATA_DIR / "pizza_steak_sushi"
TRAIN_DIR = IMAGE_DIR / "train"
TEST_DIR = IMAGE_DIR / "test"

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

BATCH_SIZE = 32

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    TRAIN_DIR, TEST_DIR, weights.transforms(), BATCH_SIZE, 0
)
train_dataloader, test_dataloader, class_names, weights.transforms()

(<torch.utils.data.dataloader.DataLoader at 0x26c157223c0>,
 ['pizza', 'steak', 'sushi'],
 ImageClassification(
     crop_size=[224]
     resize_size=[256]
     mean=[0.485, 0.456, 0.406]
     std=[0.229, 0.224, 0.225]
     interpolation=InterpolationMode.BICUBIC
 ))

# 3. Get Pretrained Model

In [16]:
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

for param in model.features.parameters():
    param.requires_grad = False

In [17]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [18]:
from torch import nn

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names), bias=True)
).to(device)
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)

In [19]:
from torchinfo import summary

summary(model= model,
        input_size=(BATCH_SIZE, 3, 64, 64),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 64, 64]      [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 64, 64]      [32, 1280, 2, 2]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 64, 64]      [32, 32, 32, 32]     --                   False
│    │    └─Conv2d (0)                                       [32, 3, 64, 64]      [32, 32, 32, 32]     (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 32, 32]     [32, 32, 32, 32]     (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 32, 32]     [32, 32, 32, 32]     --                   --
│    └─Sequential (1)                                        [32, 32, 32, 32]     [32, 

# 4. Train Model and Track Results

## 4.1 loss function and optimizer

In [20]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 4.2 Adjust `train.py` to use `SummaryWriter()`

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()
writer

可以用`SummaryWriter`的`add_scalars(main_tag: str, tag_scalar_dict: Dict)` 来记录各种值, 比如`train()`过程中的 loss 与 acc
- `main_tag`: 标签, 如`Accuracy`
- `tag_scalar_dict`: 字典, 记录不同的值, 如`{'train_loss': train_loss, 'train_accuracy': train_acc}`

使用结束后, 需要用`writer.close()`来关闭`SummaryWriter()`对象, 否则可能会出现数据丢失的情况

后续我直接在train.py进行了修改